# [양자컴퓨팅 부트캠프] Day 2: 다중 큐비트, 텐서곱, 그리고 양자 얽힘(Entanglement)

**강사**: 서상협 ((주)큐렌스) | **목표**: NumPy만을 사용하여 다중 큐비트의 상태공간, CNOT 행렬, 4대 Bell 상태, 그리고 QPU 내부 부분 측정을 바닥부터 구현하고 검증합니다.

## 📌 컴퓨터공학적 관점에서의 5대 학습 목표
1. 텐서곱(Kronecker Product, `np.kron`)의 정의와 혼합곱 성질 `(A ⊗ B)(C ⊗ D) = (AC) ⊗ (BD)` 수치 검증
2. $4 \times 4$ CNOT 게이트 행렬과 $H$ 게이트를 결합하여 다중 큐비트 알고리즘의 빌딩 블록인 Bell 상태 $\ket{\Phi^+}$ 합성
3. 선형대수학적 얽힘 판별식($c_{00}c_{11} - c_{01}c_{10} \neq 0$)을 통한 양자 우위(Quantum Advantage) 원천 증명
4. [2부 챌린지] 4대 Bell 상태($\ket{\Phi^\pm}, \ket{\Psi^\pm}$) 전수 생성 및 상호 직교 기저 검증
5. QPU 내부 부분 측정 시뮬레이션: 보조 큐비트 사영 붕괴를 통한 양자 오류 정정(QEC) 기초 원리 체득

In [ ]:
from __future__ import annotations
import numpy as np

# 기본 1큐비트 기저 및 유니터리 연산자 정의
KET_0 = np.array([[1.0], [0.0]], dtype=complex)
KET_1 = np.array([[0.0], [1.0]], dtype=complex)

GATE_I = np.eye(2, dtype=complex)
GATE_X = np.array([[0, 1], [1, 0]], dtype=complex)
GATE_Z = np.array([[1, 0], [0, -1]], dtype=complex)
GATE_H = (1.0 / np.sqrt(2.0)) * np.array([[1, 1], [1, -1]], dtype=complex)

print('1큐비트 기본 기저 및 연산자 세팅 완료!')

## [Lab 1] 텐서곱(Tensor Product)과 혼합곱 성질

두 큐비트의 상태는 단일 큐비트 공간의 텐서곱으로 확장됩니다:
$$\ket{00} = \ket{0} \otimes \ket{0} = \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix}, \quad \ket{11} = \ket{1} \otimes \ket{1} = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 1 \end{pmatrix}$$

또한 텐서곱은 매우 중요한 **혼합곱 공식(Mixed-Product Property)**을 만족합니다:
$$(A \otimes B)(C \otimes D) = (AC) \otimes (BD)$$

In [ ]:
# 2큐비트 계산 기저 생성 (np.kron)
KET_00 = np.kron(KET_0, KET_0)
KET_01 = np.kron(KET_0, KET_1)
KET_10 = np.kron(KET_1, KET_0)
KET_11 = np.kron(KET_1, KET_1)

# 혼합곱 공식 수치 검증
lhs = np.kron(GATE_H, GATE_X) @ np.kron(GATE_Z, GATE_H)
rhs = np.kron(GATE_H @ GATE_Z, GATE_X @ GATE_H)

assert np.allclose(lhs, rhs), '혼합곱 공식 불일치!'
print('[Lab 1 성공] (H ⊗ X)(Z ⊗ H) == (HZ) ⊗ (XH) 수치적 일치 검증 완료!')

## [Lab 2] $4 \times 4$ CNOT 게이트와 Bell 상태 유도

CNOT 게이트는 제어 큐비트($q_0$)가 1일 때 타깃 큐비트($q_1$)를 반전($X$)시키는 $4 \times 4$ 행렬입니다:
$$CX = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$

1부에서 손계산한 2단계 연산을 실행합니다:
$$\ket{\psi_1} = (H \otimes I)\ket{00} = \frac{\ket{00}+\ket{10}}{\sqrt{2}} \implies \ket{\Phi^+} = CX \ket{\psi_1} = \frac{\ket{00}+\ket{11}}{\sqrt{2}}$$

In [ ]:
# 4x4 CNOT 행렬 정의
GATE_CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=complex)

# Bell 회로 2단계 연산
H_I = np.kron(GATE_H, GATE_I)
psi_step1 = H_I @ KET_00
psi_bell = GATE_CNOT @ psi_step1

# 1부 손계산 결과와 1:1 대조 (오차 1e-15 이내)
expected_bell = (1.0 / np.sqrt(2.0)) * (KET_00 + KET_11)
assert np.allclose(psi_bell, expected_bell), '손계산 결과와 불일치!'
print('[Lab 2 성공] 손계산 |Phi+>와 NumPy 연산 100% 일치 확인!')
print('최종 상태벡터:', psi_bell.flatten().real)

## [Lab 3] 선형대수학적 얽힘 판별식과 양자 우위

일반적인 2큐비트 상태 $\ket{\psi} = c_{00}\ket{00} + c_{01}\ket{01} + c_{10}\ket{10} + c_{11}\ket{11}$가
개별 상태의 단순 곱 $(\alpha_0\ket{0}+\alpha_1\ket{1}) \otimes (\beta_0\ket{0}+\beta_1\ket{1})$로 인수분해(분리)될 필요충분조건은 계수 행렬의 행렬식이 0인 것입니다:
$$\det = c_{00}c_{11} - c_{01}c_{10} = 0$$

> **컴퓨터공학적 의미**: 얽힘이 없는 상태($\det = 0$)는 50큐비트라도 100개의 복소수만으로 일반 스마트폰에서 수 밀리초 만에 시뮬레이션됩니다. 오직 얽힘($\det \neq 0$)이 존재할 때만 고전 컴퓨터가 흉내 낼 수 없는 **진정한 양자 우위(Quantum Advantage)**가 발생합니다.

In [ ]:
def is_entangled(state_4d: np.ndarray, tol: float = 1e-9) -> tuple[bool, float]:
    v = state_4d.flatten()
    det = v[0] * v[3] - v[1] * v[2]
    entangled = not np.isclose(abs(det), 0.0, atol=tol)
    return entangled, float(abs(det))

# 1. Step 1 상태 (|+0>) 판별
ent1, det1 = is_entangled(psi_step1)
assert not ent1 and det1 == 0.0, 'Step 1은 분리 가능한 곱상태여야 합니다.'

# 2. 최종 Bell 상태 (|Phi+>) 판별
ent2, det2 = is_entangled(psi_bell)
assert ent2 and det2 > 0.0, 'Bell 상태는 비분리 얽힘 상태여야 합니다.'

print(f'[Lab 3 성공] 곱상태 det = {det1:.1f} (스마트폰 시뮬레이션 가능) / Bell 상태 det = {det2:.4f} (양자 우위 발생)')

## [2부 챌린지] 4대 Bell 상태 ($\ket{\Phi^\pm}, \ket{\Psi^\pm}$) 전수 합성

동일한 회로 $CNOT \cdot (H \otimes I)$에 4가지 기저 벡터를 통과시켜 4대 Bell 상태를 모두 합성하고, 이들이 4차원 공간에서 완벽히 상호 직교(Orthogonal)함을 증명합니다:

* $\ket{00} \to \ket{\Phi^+} = \frac{\ket{00}+\ket{11}}{\sqrt{2}}$
* $\ket{01} \to \ket{\Psi^+} = \frac{\ket{01}+\ket{10}}{\sqrt{2}}$
* $\ket{10} \to \ket{\Phi^-} = \frac{\ket{00}-\ket{11}}{\sqrt{2}}$
* $\ket{11} \to \ket{\Psi^-} = \frac{\ket{01}-\ket{10}}{\sqrt{2}}$

In [ ]:
inputs = {
    '|00>': KET_00,
    '|01>': KET_01,
    '|10>': KET_10,
    '|11>': KET_11
}

bell_states = {}
for name, vec in inputs.items():
    bell_states[name] = GATE_CNOT @ (H_I @ vec)

# 상호 직교성 검증 (<u|v> == 0)
phi_p = bell_states['|00>']
psi_p = bell_states['|01>']
phi_m = bell_states['|10>']
psi_m = bell_states['|11>']

assert np.isclose(np.vdot(phi_p, phi_m), 0.0)
assert np.isclose(np.vdot(phi_p, psi_p), 0.0)
assert np.isclose(np.vdot(psi_p, psi_m), 0.0)
print('[2부 챌린지 성공] 4대 Bell 기저가 모두 합성되었으며 상호 직교함이 확인되었습니다!')

## [Lab 4] QPU 내부의 부분 측정(Partial Measurement)과 조건부 제어

실제 QPU 칩 내부에서는 메인 큐비트의 중첩을 깨뜨리지 않고 계산 오류를 점검하기 위해, 보조 큐비트(Ancilla)를 얽어놓고 보조 큐비트만 부분 측정하는 **양자 오류 정정(QEC)** 기술을 사용합니다.

Bell 상태 $\ket{\Phi^+}$에서 $q_0$만 먼저 사영 측정하면:
* $q_0=0$ 관측 시 (확률 50%): 상태는 $\ket{00}$으로 투영되어 $q_1$은 즉시 $\ket{0}$으로 확정
* $q_0=1$ 관측 시 (확률 50%): 상태는 $\ket{11}$로 투영되어 $q_1$은 즉시 $\ket{1}$로 확정

In [ ]:
def simulate_partial_measurement(state_4d: np.ndarray) -> tuple[tuple[float, np.ndarray], tuple[float, np.ndarray]]:
    v = state_4d.flatten()
    p0 = float(abs(v[0])**2 + abs(v[1])**2)
    p1 = float(abs(v[2])**2 + abs(v[3])**2)
    state_q1_0 = np.array([v[0], v[1]], dtype=complex) / np.sqrt(p0)
    state_q1_1 = np.array([v[2], v[3]], dtype=complex) / np.sqrt(p1)
    return (p0, state_q1_0), (p1, state_q1_1)

(p0, s0), (p1, s1) = simulate_partial_measurement(psi_bell)

assert np.allclose(s0, [1, 0]) and np.allclose(s1, [0, 1])
print(f'- q0 = 0 관측 (확률 {p0*100:.0f}%): q1 상태는 즉시 |0> 으로 붕괴: {s0.real}')
print(f'- q0 = 1 관측 (확률 {p1*100:.0f}%): q1 상태는 즉시 |1> 로 붕괴: {s1.real}')
print('[Lab 4 성공] QPU 내부의 조건부 상태 제어 메커니즘 검증 완료!')